In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
tree = ET.parse("CDR_DevelopmentSet.BioC.xml")
root = tree.getroot()

In [3]:
documents_data = []
passages_data = []
annotations_data = []
relations_data = []

In [4]:
for document in root.findall("document"):
    
    doc_id = document.find("id").text
    
    documents_data.append({
        "document_id": doc_id
    })

In [6]:
for p_idx, passage in enumerate(document.findall("passage")):
    passage_offset = passage.find("offset").text
    passage_text = passage.find("text").text
        
    passages_data.append({
            "document_id": doc_id,
            "passage_id": p_idx,
            "passage_offset": passage_offset,
            "passage_text": passage_text
        })

In [8]:
for annotation in passage.findall("annotation"):
    
    # ID as attribute
    ann_id = annotation.attrib.get("id")
    
    # Text (safe extraction)
    text_tag = annotation.find("text")
    ann_text = text_tag.text if text_tag is not None else None
    
    # Location
    location = annotation.find("location")
    
    start = location.attrib.get("offset") if location is not None else None
    length = location.attrib.get("length") if location is not None else None
    
    # Extract all infon metadata dynamically
    infon_data = {}
    for infon in annotation.findall("infon"):
        key = infon.attrib.get("key")
        value = infon.text
        infon_data[key] = value
    
    annotations_data.append({
        "document_id": doc_id,
        "passage_id": p_idx,
        "annotation_id": ann_id,
        "entity_text": ann_text,
        "start_offset": start,
        "length": length,
        **infon_data
    })

In [10]:
for relation in document.findall("relation"):
    
    # ID is attribute
    rel_id = relation.attrib.get("id")
    
    # Extract infon metadata dynamically
    infon_data = {}
    for infon in relation.findall("infon"):
        key = infon.attrib.get("key")
        value = infon.text
        infon_data[key] = value
    
    # Extract nodes safely
    nodes = relation.findall("node")
    
    arg1 = None
    arg2 = None
    
    if len(nodes) >= 1:
        arg1 = nodes[0].attrib.get("refid")
    if len(nodes) >= 2:
        arg2 = nodes[1].attrib.get("refid")
    
    relations_data.append({
        "document_id": doc_id,
        "relation_id": rel_id,
        "arg1": arg1,
        "arg2": arg2,
        **infon_data
    })

In [11]:
df_documents = pd.DataFrame(documents_data)
df_passages = pd.DataFrame(passages_data)
df_annotations = pd.DataFrame(annotations_data)
df_relations = pd.DataFrame(relations_data)

In [12]:
df_documents.to_csv("documents.csv", index=False)
df_passages.to_csv("passages.csv", index=False)
df_annotations.to_csv("annotations.csv", index=False)
df_relations.to_csv("relations.csv", index=False)

In [13]:
final_df = df_annotations.merge(df_passages, 
                                on=["document_id", "passage_id"])

final_df.to_csv("CDR_full_flat.csv", index=False)